In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from sklearn import feature_extraction, model_selection, naive_bayes, metrics, svm
from IPython.display import Image
import warnings
import seaborn as sns
warnings.filterwarnings("ignore")
%matplotlib inline

In [2]:
df.info()

NameError: name 'df' is not defined

In [ ]:
df = pd.read_csv("../dataset/spam.csv", encoding="latin-1")
df.head(n=10)

In [ ]:
sns.countplot(df, x='v1', hue='v1')
plt.title("Distribution Category")

In [ ]:
data = df['v1'].value_counts()
data

In [ ]:
plt.pie(data, labels=['ham', 'spam'], colors=sns.color_palette('bright'), autopct='%.0f%%')
plt.show()

In [ ]:
count1 = Counter(" ".join(df[df['v1'] == 'ham']['v2']).split()).most_common(20)
df1 = pd.DataFrame.from_dict(count1)
df1 = df1.rename(columns={0: "word in non-spam", 1:"count"})
df1

In [ ]:
count2 = Counter(" ".join(df[df['v1'] == 'spam']['v2']).split()).most_common(20)
df2 = pd.DataFrame.from_dict(count2)
df2 = df2.rename(columns={0: "word in spam", 1: "count"})
df2

In [ ]:
sns.barplot(data=df1, x=df1['word in non-spam'], y=df1['count'])
plt.xticks(rotation=90)
plt.title("Distribution word non-spam")
plt.show()

In [ ]:
sns.barplot(data=df2, x=df2['word in spam'], y=df2['count'])
plt.xticks(rotation=90)
plt.show()

In [ ]:
transform = feature_extraction.text.CountVectorizer(stop_words='english')
X = transform.fit_transform(df['v2'])
np.shape(X)

In [ ]:
df['v1'] = df['v1'].map({'spam':1, 'ham':0})
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, df['v1'], test_size=0.33, random_state=42)

In [ ]:
list_alpha = np.arange(1/100000, 20, 0.11)
score_train = np.zeros(len(list_alpha))
score_test = np.zeros(len(list_alpha))
recall_test = np.zeros(len(list_alpha))
precision_test = np.zeros(len(list_alpha))

count = 0

for alpha in list_alpha:
    bayes = naive_bayes.MultinomialNB(alpha=alpha)
    bayes.fit(X_train, y_train)
    score_train[count] = bayes.score(X_train, y_train)
    score_test[count] = bayes.score(X_test, y_test)
    recall_test[count] = metrics.recall_score(y_test, bayes.predict(X_test))
    precision_test[count] = metrics.precision_score(y_test, bayes.predict(X_test))
    count = count+1

In [ ]:
matrix = np.matrix(np.c_[list_alpha, score_train, score_test, recall_test, precision_test])
models = pd.DataFrame(data = matrix, columns=['alpha', 'Train Accuracy', 'Test Accuracy', 'Test Recall', 'Test Precision'])
models.head(n=10)

In [ ]:
best_index = models["Test Precision"].idxmax()
models.iloc[best_index, :]

In [ ]:
models[models['Test Precision'] == 1].head(n=5)

In [ ]:
best_index = models[models['Test Precision'] == 1]['Test Accuracy'].idxmax()
bayes = naive_bayes.MultinomialNB(alpha=list_alpha[best_index])
bayes.fit(X_train, y_train)
models.iloc[best_index, :]

In [ ]:
m_confusion_test = metrics.confusion_matrix(y_test, bayes.predict(X_test))
pd.DataFrame(data = m_confusion_test, columns=['Predicted 0', 'Predicted 1'], index= ['Actual 0', 'Actual 1'])